# 3.3 Advanced RAG - HybridSearch Engine

## Project Role: HybridSearcher is the core retrieval module for the final Agent

In [ ]:
import sys, os
sys.path.insert(0, "..")
from config import get_client; client = get_client()
from agent_project import *
from agent_project.tools import create_default_registry
print(f"LLM: {client.name} | modules loaded")


In [ ]:
from agent_project.hybrid_search import HybridSearcher
searcher = HybridSearcher()

DOCS = [
    "BM25 excels at exact keyword matching like error codes and product IDs",
    "Vector search handles semantic similarity synonyms and cross-language queries",
    "Hybrid search combines BM25 and vector for best accuracy and recall",
    "RRF fusion merges rankings without tuning using reciprocal rank positions",
    "Cross-Encoder reranking is 10-50x slower than Bi-Encoder but more accurate",
    "Query rewriting expands user queries into multiple variants for better recall",
    "Self-RAG evaluates retrieval quality and triggers fallback when needed",
    "HyDE generates hypothetical document vectors to bridge semantic gaps",
    "DeepSeek V4 achieves 80.6% on SWE-bench code generation benchmark",
    "Qwen3.7-Max has 1M context handling entire books at once",
]
searcher.index(DOCS)
print(f"Hybrid index built: {len(DOCS)} documents")
print(f"Components: BM25 + Vector + RRF + Reranker + Self-Assessment")


In [ ]:
print("===== Hybrid vs Vector Comparison =====\n")
queries = [
    ("BM25 algorithm details", "exact keyword match"),
    ("How to find similar documents?", "semantic search"),
    ("Best retrieval for RAG?", "hybrid scenario"),
]
for q, desc in queries:
    print(f"Query [{desc}]: {q}")
    hr = searcher.search(q, top_k=3, use_hybrid=True, use_rerank=True)
    vr = searcher.search(q, top_k=3, use_hybrid=False, use_rerank=False)
    print(f"  Hybrid: {[r["doc_id"] for r in hr]}  scores={[r["score"] for r in hr]}")
    print(f"  Vector: {[r["doc_id"] for r in vr]}  scores={[r["score"] for r in vr]}")
    print()


In [ ]:
print("===== Self-Assessment Test =====\n")
tests = [("What is BM25?", True), ("Weather forecast?", False)]
for q, expect_good in tests:
    r = searcher.search(q, top_k=5)
    a = searcher.self_assess(r)
    status = "OK" if ((a["grade"] in "AB") == expect_good) else "CHECK"
    print(f"  [{status}] {q}: grade={a["grade"]}, action={a["action"]}")
print("\nHybridSearcher ready for final project!")
